[![Roboflow Notebooks](https://media.roboflow.com/notebooks/template/bannertest2-2.png?ik-sdk-version=javascript-1.4.3&updatedAt=1672932710194)](https://github.com/roboflow/notebooks)

# How to Add ReID to Trackers

BoT-SORT can fuse visual ReID embeddings with IoU during association. In this
notebook you will enable appearance ReID with the [`reid`](https://github.com/roboflow/re-ID)
package, then run BoT-SORT + ReID on MOT17 val-half using YOLOX detections and
TrackEval metrics.

For threshold selection and MOT17 / SoccerNet results, see the
[ReID appearance guide](https://trackers.roboflow.com/develop/learn/reid/).


## Setup


### Check GPU availability

Let's make sure that we have access to GPU. We can use `nvidia-smi` command to do
that. In case of any problems navigate to `Runtime` -> `Change runtime type` ->
`Hardware accelerator`, set it to `GPU`, and then click `Save`.


In [9]:
!nvidia-smi

Thu Jul 30 15:57:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P0             27W /   70W |    1123MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Install dependencies

Install Trackers with the ReID extra. Appearance ReID is not on PyPI trackers yet,
so this notebook pins the feature branch. After release, switch to
`trackers[reid]==2.6.0`.

You may see dependency conflict warnings in Google Colab. This is expected for the
preinstalled Google Colab environment and does not affect functionality.


In [10]:
!pip install -q matplotlib gdown
!pip install -q "trackers[reid] @ git+https://github.com/roboflow/trackers.git@feat/core/reid-consume-reid-package"


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


### Download the dataset

Download MOT17 val ground truth and frames with the Trackers CLI, then fetch the
YOLOX val detections. YOLOX frame IDs
are remapped to `1...N` so they align with MOT frame indexing.


In [11]:
from pathlib import Path

REPO_ROOT = Path("..")

VAL_SEQUENCES = [
    "MOT17-02-FRCNN",
    "MOT17-04-FRCNN",
    "MOT17-05-FRCNN",
    "MOT17-09-FRCNN",
    "MOT17-10-FRCNN",
    "MOT17-11-FRCNN",
    "MOT17-13-FRCNN",
]

MOT17_VAL = REPO_ROOT / "mot17" / "val"
YOLOX_DIR = REPO_ROOT / "MOT17_yolox_dets"
YOLOX_VAL_DIR = YOLOX_DIR / "val"
YOLOX_ZIP = YOLOX_DIR / "yolox_detections_MOT17.zip"
YOLOX_GDRIVE_ID = "1BuXtPWf8QbPU_y1i2xY2IbTE-rj3l6qT"
OUTPUT_ROOT = REPO_ROOT / "trackers_reid_outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

!trackers download mot17 --split val --asset annotations,frames -o {REPO_ROOT}

!mkdir -p {YOLOX_DIR}
!gdown {YOLOX_GDRIVE_ID} -O {YOLOX_ZIP}
!unzip -qo {YOLOX_ZIP} -d {YOLOX_DIR}


[download] mot17:val:annotations
  using cached mot17-val-annotations.zip
[extract] mot17:val:annotations
[done] mot17:val:annotations
[download] mot17:val:frames
  using cached mot17-val-frames.zip
[extract] mot17:val:frames
[done] mot17:val:frames
Downloading...
From: https://drive.google.com/uc?id=1BuXtPWf8QbPU_y1i2xY2IbTE-rj3l6qT
To: /MOT17_yolox_dets/yolox_detections_MOT17.zip
100% 6.35M/6.35M [00:00<00:00, 19.6MB/s]


In [12]:
def yolox_det_path(seq: str) -> Path:
    return YOLOX_VAL_DIR / f"{seq.replace('-FRCNN', '')}_val.txt"


SEQUENCE_PATHS: dict[str, dict] = {}
for seq in VAL_SEQUENCES:
    gt = MOT17_VAL / seq / "gt" / "gt.txt"
    img = MOT17_VAL / seq / "img1"
    det = yolox_det_path(seq)
    if not (gt.is_file() and img.is_dir() and det.is_file()):
        print(f"  skip {seq}: missing gt, img1, or YOLOX det")
        continue
    n_frames = len(list(img.glob("*.jpg")))
    SEQUENCE_PATHS[seq] = {"gt": gt, "img": img, "det": det, "n_frames": n_frames}
    print(f"  {seq}: {n_frames} frames")

ACTIVE_SEQUENCES = list(SEQUENCE_PATHS)
if not ACTIVE_SEQUENCES:
    raise RuntimeError("No sequences ready - re-run the download cell above.")

SEQMAP_PATH = OUTPUT_ROOT / "MOT17-val.txt"
SEQMAP_PATH.write_text("name\n" + "\n".join(ACTIVE_SEQUENCES) + "\n")
print(f"\n{len(ACTIVE_SEQUENCES)} sequences ready")


  MOT17-02-FRCNN: 299 frames
  MOT17-04-FRCNN: 524 frames
  MOT17-05-FRCNN: 418 frames
  MOT17-09-FRCNN: 262 frames
  MOT17-10-FRCNN: 326 frames
  MOT17-11-FRCNN: 449 frames
  MOT17-13-FRCNN: 374 frames

7 sequences ready


## Load ReID model

Load a MOT17 FastReID SBS50 encoder from the `reid` package. For other checkpoints
(for example OSNet on MSMT17), pass a different model id to
`ReIDModel.from_pretrained(...)`. See the
[`reid` training guide](https://github.com/roboflow/re-ID/blob/main/docs/learn/train.md).


In [13]:
import warnings

import cv2
import matplotlib.pyplot as plt
import numpy as np
import supervision as sv
import torch
from reid import FASTREID_MOT17_SBS50, ReIDModel

from trackers import BoTSORTTracker
from trackers.eval import evaluate_mot_sequences
from trackers.io.frames import load_mot_frame_image
from trackers.io.mot import _MOTOutput, load_mot_file

warnings.filterwarnings("ignore")

device = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()} | {device}")

# Use another reid model id here to try OSNet or a custom checkpoint.
REID_ENCODER = FASTREID_MOT17_SBS50
REID_APPEARANCE_THRESHOLD = 0.2
reid_model = ReIDModel.from_pretrained(REID_ENCODER)

print(f"Encoder: {REID_ENCODER}  |  appearance_threshold: {REID_APPEARANCE_THRESHOLD}")
print(reid_model.preprocessing.describe())


PyTorch 2.11.0+cu128 | CUDA True | Tesla T4
Encoder: fastreid_mot17_sbs50  |  appearance_threshold: 0.2
ReIDPreprocessing(resize=384x128 [stretch, bilinear], BGR→RGB, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))


## Run BoT-SORT with ReID

Create a `BoTSORTTracker` with a `reid_model` and pass the current frame to
`update()` so embeddings can be extracted.


In [ ]:
def load_yolox_dets(det_path: Path) -> dict[int, sv.Detections]:
    """Load YOLOX dets (`frame,x1,y1,x2,y2,score`) as 1-based frame -> Detections."""
    rows: list[tuple[int, float, float, float, float, float]] = []
    with det_path.open() as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) < 6:
                continue
            frame, x1, y1, x2, y2, score = map(float, parts[:6])
            if score > 0:
                rows.append((int(frame), x1, y1, x2, y2, score))
    if not rows:
        return {}
    min_frame = min(frame for frame, *_ in rows)
    offset = min_frame - 1 if min_frame > 1 else 0
    by_frame: dict[int, list[list[float]]] = {}
    for frame, x1, y1, x2, y2, score in rows:
        by_frame.setdefault(frame - offset, []).append([x1, y1, x2, y2, score])
    return {
        frame: sv.Detections(
            xyxy=np.asarray(boxes, dtype=np.float32)[:, :4],
            confidence=np.asarray(boxes, dtype=np.float32)[:, 4],
        )
        for frame, boxes in by_frame.items()
    }


pred_dir = OUTPUT_ROOT / "botsort_reid" / "preds"
pred_dir.mkdir(parents=True, exist_ok=True)

for seq in ACTIVE_SEQUENCES:
    spec = SEQUENCE_PATHS[seq]
    dets = load_yolox_dets(spec["det"])
    images = sorted(spec["img"].glob("*.jpg"))
    tracker = BoTSORTTracker(
        enable_cmc=True,
        reid_model=reid_model,
        reid_ema_alpha=0.9,
        appearance_threshold=REID_APPEARANCE_THRESHOLD,
    )
    with _MOTOutput(pred_dir / f"{seq}.txt") as out:
        for frame_idx in range(1, spec["n_frames"] + 1):
            frame = cv2.imread(str(images[frame_idx - 1]))
            tracked = tracker.update(dets.get(frame_idx, sv.Detections.empty()), frame)
            if tracked.tracker_id is not None:
                tracked = tracked[tracked.tracker_id != -1]
            out.write(frame_idx, tracked)
    print(f"  {seq}: {spec['n_frames']} frames")

result_reid = evaluate_mot_sequences(
    gt_dir=MOT17_VAL,
    tracker_dir=pred_dir,
    seqmap=SEQMAP_PATH,
    metrics=["CLEAR", "HOTA", "Identity"],
)
agg = result_reid.aggregate
print(
    f"BoT-SORT + ReID: HOTA {agg.HOTA.HOTA * 100:6.2f}  "
    f"MOTA {agg.CLEAR.MOTA * 100:6.2f}  "
    f"IDF1 {agg.Identity.IDF1 * 100:6.2f}  "
    f"IDSW {agg.CLEAR.IDSW}"
)


## Appearance distance histogram

Encoder diagnostic on MOT17 val **GT crops** (no detector). Pair sampling matches
what BoT-SORT association can see: one video at a time, within the lost-track
horizon.

- Axis: `d_app = 0.5 * (1 - cos)` (BoT-SORT `embedding_distance / 2`)
- Positives: same ID, same sequence, `1 <= |frame gap| <= MAX_FRAME_GAP`
- Negatives: different ID, same sequence, `|frame gap| <= MAX_FRAME_GAP`
- θ lines: 0.25 (default) and `REID_APPEARANCE_THRESHOLD`

`MAX_FRAME_GAP` defaults to 30 (BoT-SORT `lost_track_buffer` at 30 FPS).


In [ ]:
from collections import defaultdict

N_INTRA, N_INTER = 5000, 10000
MIN_FRAME_GAP, MAX_FRAME_GAP = 1, 30  # ~lost_track_buffer @ 30 FPS
CANDIDATE_THETAS = (0.20, 0.25)
DAPP_BINS = np.linspace(0.0, 1.0, 51)


def pedestrian_detections(gt_frame) -> sv.Detections:
    keep = (gt_frame.confidences > 0) & (gt_frame.classes == 1)
    if not np.any(keep):
        return sv.Detections.empty()
    return sv.Detections(
        xyxy=sv.xywh_to_xyxy(gt_frame.boxes[keep]).astype(np.float32),
        tracker_id=gt_frame.ids[keep].astype(int),
    )


def collect_gt_embeddings(model: ReIDModel, sequences: list[str], frame_stride: int = 1):
    embeddings, labels, frame_ids, seq_ids = [], [], [], []
    label_by_key: dict[str, int] = {}

    for sid, seq in enumerate(sequences):
        spec = SEQUENCE_PATHS[seq]
        gt_by_frame = load_mot_file(spec["gt"])
        images = sorted(spec["img"].glob("*.jpg"))
        for frame_idx in range(1, spec["n_frames"] + 1, frame_stride):
            gt = gt_by_frame.get(frame_idx)
            if gt is None:
                continue
            dets = pedestrian_detections(gt)
            if len(dets) == 0:
                continue
            bgr = cv2.imread(str(images[frame_idx - 1]))
            if bgr is None:
                continue
            feats = model.extract_features(dets, bgr)
            for i, tid in enumerate(dets.tracker_id):
                key = f"{seq}_{int(tid)}"
                label_by_key.setdefault(key, len(label_by_key))
                embeddings.append(feats[i])
                labels.append(label_by_key[key])
                frame_ids.append(frame_idx)
                seq_ids.append(sid)

    if not embeddings:
        raise RuntimeError("No GT embeddings (conf>0, class==1 pedestrians).")
    return (
        np.stack(embeddings),
        np.asarray(labels, dtype=np.int64),
        np.asarray(frame_ids, dtype=np.int64),
        np.asarray(seq_ids, dtype=np.int64),
    )


def _sample_pair_distances(normed: np.ndarray, pairs: list[tuple[int, int]], n: int, rng) -> np.ndarray:
    pick = np.asarray(pairs, dtype=np.int64)[rng.integers(len(pairs), size=n)]
    return 0.5 * (1.0 - np.einsum("ij,ij->i", normed[pick[:, 0]], normed[pick[:, 1]]))


def sample_association_local_distances(
    embeddings: np.ndarray,
    gt_ids: np.ndarray,
    *,
    frame_ids: np.ndarray,
    seq_ids: np.ndarray,
    n_intra: int = N_INTRA,
    n_inter: int = N_INTER,
    min_frame_gap: int = MIN_FRAME_GAP,
    max_frame_gap: int = MAX_FRAME_GAP,
    seed: int = 0,
) -> tuple[np.ndarray, np.ndarray]:
    """Same-video pairs within max_frame_gap (tracker association horizon)."""
    normed = embeddings / (np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-12)
    rng = np.random.default_rng(seed)

    by_seq_id: dict[tuple[int, int], list[int]] = defaultdict(list)
    by_seq: dict[int, list[int]] = defaultdict(list)
    for idx in range(len(gt_ids)):
        by_seq_id[(int(seq_ids[idx]), int(gt_ids[idx]))].append(idx)
        by_seq[int(seq_ids[idx])].append(idx)

    pos_pairs: list[tuple[int, int]] = []
    for idxs in by_seq_id.values():
        for a, ia in enumerate(idxs):
            fa = int(frame_ids[ia])
            for ib in idxs[a + 1 :]:
                gap = abs(fa - int(frame_ids[ib]))
                if min_frame_gap <= gap <= max_frame_gap:
                    pos_pairs.append((ia, ib))
    if not pos_pairs:
        raise ValueError(f"No same-ID pairs with {min_frame_gap}<=|Delta frame|<={max_frame_gap}.")

    neg_pairs: list[tuple[int, int]] = []
    max_pool = max(n_inter * 40, 50_000)
    for idxs in by_seq.values():
        idxs_sorted = sorted(idxs, key=lambda i: int(frame_ids[i]))
        frames = [int(frame_ids[i]) for i in idxs_sorted]
        for a, ia in enumerate(idxs_sorted):
            fa, pid_a = frames[a], int(gt_ids[ia])
            b = a + 1
            while b < len(idxs_sorted) and frames[b] - fa <= max_frame_gap:
                ib = idxs_sorted[b]
                if int(gt_ids[ib]) != pid_a:
                    neg_pairs.append((ia, ib))
                    if len(neg_pairs) >= max_pool:
                        break
                b += 1
            if len(neg_pairs) >= max_pool:
                break
        if len(neg_pairs) >= max_pool:
            break
    if not neg_pairs:
        raise ValueError(f"No different-ID pairs with |Delta frame|<={max_frame_gap} in-sequence.")

    return (
        _sample_pair_distances(normed, pos_pairs, n_intra, rng),
        _sample_pair_distances(normed, neg_pairs, n_inter, rng),
    )


emb_gt, gt_ids, frame_ids, seq_ids = collect_gt_embeddings(reid_model, ACTIVE_SEQUENCES)
print(f"GT pool: {len(emb_gt)} crops, {len(np.unique(gt_ids))} ids, {len(np.unique(seq_ids))} sequences")

intra, inter = sample_association_local_distances(
    emb_gt, gt_ids, frame_ids=frame_ids, seq_ids=seq_ids
)
print(f"pairs: same-ID={len(intra)}  diff-ID={len(inter)}  (same seq, |Delta frame|<={MAX_FRAME_GAP})")
print(
    f"d_app means: same-ID={intra.mean():.3f}  diff-ID={inter.mean():.3f}  "
    f"gap={inter.mean() - intra.mean():.3f}  "
    f"same-ID p95={np.quantile(intra, 0.95):.3f}"
)

fig, ax = plt.subplots(figsize=(8, 4.5))
for values, label, color in (
    (intra, f"same-ID (n={len(intra)})", "#3366CC"),
    (inter, f"different-ID (n={len(inter)})", "#DC3912"),
):
    ax.hist(values, bins=DAPP_BINS, weights=np.full(len(values), 1.0 / len(values)), alpha=0.65, label=label, color=color)
ax.axvline(0.25, color="#666666", ls=":", lw=1.5, label="θ=0.25 (default)")
ax.axvline(REID_APPEARANCE_THRESHOLD, color="#111111", ls="--", lw=1.8, label=f"θ={REID_APPEARANCE_THRESHOLD:.2f} (selected)")
ax.set(xlabel=r"$0.5\cdot$ cosine distance", ylabel="probability", title=f"{REID_ENCODER} on MOT17 val GT", xlim=(0.0, 0.6))
ax.legend(frameon=False, fontsize=9)
ax.grid(True, alpha=0.25)
fig.tight_layout()
plt.show()

print(f"{'θ':>6}  {'same-ID < θ':>12}  {'diff-ID < θ':>12}")
for theta in CANDIDATE_THETAS:
    note = "  <- selected" if abs(theta - REID_APPEARANCE_THRESHOLD) < 1e-9 else ("  <- paper" if abs(theta - 0.25) < 1e-9 else "")
    print(f"{theta:6.2f}  {100 * float(np.mean(intra < theta)):11.1f}%  {100 * float(np.mean(inter < theta)):11.1f}%{note}")


## Results

Score BoT-SORT + ReID on MOT17 val-half. Reference numbers below come from the
[MOT17 re-ID study](https://www-sop.inria.fr/members/Francois.Bremond/Postscript/Tomasz__SCCAI_2025.pdf)
(Tables 8 and 13) and the [BoT-SORT paper](https://arxiv.org/abs/2206.14651) (Table 1).

| Source | Config | HOTA | IDF1 |
|---|---|---:|---:|
| MOT17 re-ID study | No re-ID | 68.43 | 80.92 |
| MOT17 re-ID study | FastReID, th=0.2 | 68.95 | 81.98 |
| BoT-SORT paper | BoT-SORT | 69.11 | 81.53 |
| BoT-SORT paper | BoT-SORT + ReID | 69.17 | 82.07 |


In [ ]:
REF_NO_REID = {"hota": 68.43, "idf1": 80.92}
REF_REID = {"hota": 68.95, "idf1": 81.98}

agg = result_reid.aggregate
hota = agg.HOTA.HOTA * 100
mota = agg.CLEAR.MOTA * 100
idf1 = agg.Identity.IDF1 * 100
idsw = agg.CLEAR.IDSW

print(f"{'Config':<28}  {'HOTA':>6}  {'MOTA':>6}  {'IDF1':>6}  {'IDSW':>5}")
print("-" * 58)
print(f"{'BoT-SORT + ReID (this run)':<28}  {hota:6.2f}  {mota:6.2f}  {idf1:6.2f}  {idsw:5d}")
print(f"{'MOT17 study (no ReID)':<28}  {REF_NO_REID['hota']:6.2f}  {'-':>6}  {REF_NO_REID['idf1']:6.2f}  {'-':>5}")
print(f"{'MOT17 study (FastReID)':<28}  {REF_REID['hota']:6.2f}  {'-':>6}  {REF_REID['idf1']:6.2f}  {'-':>5}")
print(
    f"\nvs MOT17 study FastReID: "
    f"HOTA {hota - REF_REID['hota']:+.2f}, IDF1 {idf1 - REF_REID['idf1']:+.2f}"
)

print(f"\n{'Sequence':<18}  {'HOTA':>6}  {'AssA':>6}  {'IDF1':>6}  {'IDSW':>5}")
print("-" * 48)
for seq in ACTIVE_SEQUENCES:
    s = result_reid.sequences[seq]
    print(
        f"{seq:<18}  {s.HOTA.HOTA * 100:6.2f}  {s.HOTA.AssA * 100:6.2f}  "
        f"{s.Identity.IDF1 * 100:6.2f}  {s.CLEAR.IDSW:5d}"
    )


## Sample tracked frames

Plot a few frames from one sequence with BoT-SORT + ReID track IDs.


In [ ]:
VIZ_SEQ = ACTIVE_SEQUENCES[0]  # override with e.g. "MOT17-02-FRCNN"
VIZ_FRAMES = (1, 30, 60, 90)

mot = load_mot_file(pred_dir / f"{VIZ_SEQ}.txt")
img_dir = SEQUENCE_PATHS[VIZ_SEQ]["img"]
box_ann = sv.BoxAnnotator(thickness=2, color_lookup=sv.ColorLookup.TRACK)
label_ann = sv.LabelAnnotator(
    text_color=sv.Color.BLACK,
    text_scale=0.5,
    color_lookup=sv.ColorLookup.TRACK,
)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, frame_idx in zip(axes.ravel(), VIZ_FRAMES):
    frame = load_mot_frame_image(img_dir, frame_idx)
    data = mot.get(frame_idx)
    if data is None or len(data.ids) == 0:
        scene = frame
    else:
        dets = sv.Detections(
            xyxy=sv.xywh_to_xyxy(data.boxes).astype(np.float32),
            tracker_id=data.ids.astype(int),
        )
        scene = box_ann.annotate(frame.copy(), dets)
        scene = label_ann.annotate(scene, dets, labels=[str(int(i)) for i in dets.tracker_id])
    ax.imshow(scene[:, :, ::-1])
    ax.set_title(f"{VIZ_SEQ}  frame {frame_idx}")
    ax.axis("off")

fig.suptitle("BoT-SORT + ReID", y=1.01)
fig.tight_layout()
plt.show()

You just ran BoT-SORT with appearance ReID on MOT17. Nice work!

Trackers makes it easy to mix and match multi-object tracking algorithms with your
favorite detection backends. Appearance association is optional: install
`trackers[reid]`, pass a `reid.ReIDModel`, and supply `frame=` to `update()`.

Ready to go deeper? Explore the [ReID appearance guide](https://trackers.roboflow.com/develop/learn/reid/),
the [`reid` package](https://github.com/roboflow/re-ID), or the Trackers
[documentation](https://trackers.roboflow.com/develop/) and
[GitHub](https://github.com/roboflow/trackers).

Got feedback or ideas? Open an issue on
[GitHub Issues](https://github.com/roboflow/trackers/issues).
